# Clasificador de Rango Etario V4 — UTKFace (balanceado por edad y etnia)

Clasificación multiclase (5 clases) a partir de imágenes de rostros, implementado en **PyTorch** y **TensorFlow/Keras**.

**Segmentos etarios:**

| Clase | Rango | Etiqueta |
|---|---|---|
| 0 | 0–18 | joven |
| 1 | 19–30 | joven-adulto |
| 2 | 31–45 | adulto |
| 3 | 46–60 | semi-viejo |
| 4 | 61+ | viejo |

> ## 📌 Banco de entrenamiento: carpeta única UTKFace (balanceado por EDAD y ETNIA)
>
> Este notebook entrena leyendo **directamente** desde la carpeta original descargada, sin subcarpetas por clase:
> - Carpeta: `./UTKFace_data/UTKFace`
> - El segmento etario y la etnia se calculan en Python a partir del **nombre de archivo** (`[edad]_[género]_[raza]_[fecha].jpg`).
> - **Doble balanceo:** para evitar que el modelo aprenda a distinguir edad a partir de rasgos étnicos, se cuenta la cantidad de imágenes de cada combinación (segmento, etnia) y se toma el **mínimo global** entre las 20 combinaciones (5 segmentos × 4 etnias) como `cap`. Ese mismo `cap` se usa para **todas** las combinaciones: mismo N° de imágenes por etnia en cada segmento, y mismo N° total de imágenes en cada segmento.
> - Códigos de etnia UTKFace usados: `White, Black, Asian, Indian`. La categoría **`Others` se excluye completamente** del entrenamiento (agrupa etnias heterogéneas, incluidos hispanos/latinos, y no es una categoría étnica consistente para balancear).
>
> 👉 Otros notebooks: `clasificador_edad_utkface.ipynb` (5 clases por rango clásico), `clasificador_edad_por_decada.ipynb` (9 clases por década) y `clasificador_edad_utkface_RESULTADOS_completo.ipynb` (5 clases, UTKFace completo desbalanceado).

## 0. Descarga del dataset (UTKFace)

**Opción A — Kaggle API (recomendado):**

```bash
pip install kaggle
# 1. Crea una cuenta en kaggle.com si no tienes
# 2. Ve a kaggle.com/settings -> API -> "Create New Token" (descarga kaggle.json)
# 3. Coloca kaggle.json en ~/.kaggle/kaggle.json (Linux/Mac) o C:\Users\<usuario>\.kaggle\kaggle.json (Windows)

kaggle datasets download -d jangedoo/utkface-new
unzip utkface-new.zip -d ./UTKFace_data
```

**Opción B — Descarga manual:**

1. Ve a https://www.kaggle.com/datasets/jangedoo/utkface-new
2. Descarga el .zip (botón Download)
3. Descomprime y deja la carpeta de imágenes en `./UTKFace_data/UTKFace/` (debe contener directamente los .jpg, sin subcarpetas extra)

**Estructura esperada:**
```
UTKFace_data/
  UTKFace/
    1_0_0_20161219140623097.jpg
    2_1_3_20170109150557335.jpg
    ...
```

El nombre de cada archivo sigue el formato `[edad]_[género]_[raza]_[fecha].jpg`. Usaremos los campos `edad` y `raza`.

In [ ]:
!pip install pandas matplotlib seaborn scikit-learn torch tensorflow pillow

In [ ]:
# --- Configuración general ---
import os
import re
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

DATA_DIR = "./UTKFace_data/UTKFace"     # carpeta única con todas las imágenes (sin subcarpetas por clase)
IMG_SIZE = 64                          # tamaño al que se redimensionan las imágenes
BATCH_SIZE = 32
SEED = 42

np.random.seed(SEED)

In [ ]:
# --- Definición de los segmentos etarios y utilidades ---
CLASS_NAMES = ["joven", "joven-adulto", "adulto", "semi-viejo", "viejo"]

# Códigos de etnia de UTKFace (campo 3 del nombre de archivo).
# 'Others' (código 4) se EXCLUYE del entrenamiento: agrupa etnias muy distintas entre sí
# (incluye hispanos/latinos) y no aporta una categoría étnica consistente para el balanceo.
RACE_NAMES = {0: "White", 1: "Black", 2: "Asian", 3: "Indian"}

def age_to_segment(age: int) -> int:
    """Convierte una edad entera al índice de segmento (0-4) según los nuevos rangos."""
    if age <= 18:
        return 0  # joven
    elif age <= 30:
        return 1  # joven-adulto
    elif age <= 45:
        return 2  # adulto
    elif age <= 60:
        return 3  # semi-viejo
    else:
        return 4  # viejo

def parse_age_from_filename(filename: str):
    """Extrae la edad desde un nombre de archivo tipo '23_0_1_2017...jpg'. Devuelve None si no calza el formato."""
    base = os.path.basename(filename)
    match = re.match(r"^(\d+)_\d+_\d+_.+\.jpg$", base)
    if match is None:
        return None
    return int(match.group(1))

def parse_race_from_filename(filename: str):
    """Extrae el código de etnia del nombre '[edad]_[genero]_[raza]_[fecha].jpg'.
    Devuelve None si no calza o si la etnia es 'Others' (excluida)."""
    parts = os.path.basename(filename).split("_")
    try:
        race = int(parts[2])
        return race if race in RACE_NAMES else None
    except (ValueError, IndexError):
        return None

In [ ]:
# --- Construcción del dataframe desde la carpeta única UTKFace ---
# El segmento etario y la etnia se calculan parseando el NOMBRE DE ARCHIVO (no hay subcarpetas).
# La etnia 'Others' queda excluida (parse_race_from_filename la descarta).
# Balanceo doble: se cuenta cada combinación (segmento, etnia) y se usa el MÍNIMO GLOBAL entre
# todas las combinaciones como cap único -> mismo N° de imágenes por etnia en cada segmento.
all_fps = glob.glob(os.path.join(DATA_DIR, "*.jpg"))
if len(all_fps) == 0:
    raise FileNotFoundError(
        f"No hay imágenes en '{DATA_DIR}'. Revisa que la carpeta UTKFace exista con los .jpg "
        "directamente adentro (ver celda de descarga del dataset)."
    )

records_by_bucket = {(s, r): [] for s in range(len(CLASS_NAMES)) for r in RACE_NAMES}
skipped = 0
for fp in all_fps:
    age = parse_age_from_filename(fp)
    race = parse_race_from_filename(fp)
    if age is None or race is None:
        skipped += 1
        continue
    records_by_bucket[(age_to_segment(age), race)].append(fp)

print(f"Imágenes totales encontradas: {len(all_fps)}  |  Descartadas (nombre inválido o etnia 'Others'/desconocida): {skipped}")
print("Disponibles por (segmento, etnia) antes de balancear:")
counts_tab = pd.DataFrame(
    {RACE_NAMES[r]: [len(records_by_bucket[(s, r)]) for s in range(len(CLASS_NAMES))] for r in RACE_NAMES},
    index=CLASS_NAMES,
)
print(counts_tab)

cap = min(len(fps) for fps in records_by_bucket.values())
print(f"\nCap global (mínimo entre las {len(records_by_bucket)} combinaciones segmento×etnia): {cap}")

rng = np.random.RandomState(SEED)
records = []
for (label_idx, race), fps in records_by_bucket.items():
    sampled = rng.choice(fps, size=cap, replace=False)
    for fp in sampled:
        records.append({"filepath": fp, "label": label_idx, "race": race})

df = pd.DataFrame(records, columns=["filepath", "label", "race"])
print(f"\nImágenes usadas para entrenar (balanceado por edad y etnia): {len(df)}")
print("Por clase:")
print(df["label"].value_counts().sort_index().rename(index=lambda i: CLASS_NAMES[i]))
df.head()

In [ ]:
# --- Distribución de clases ---
counts = df["label"].value_counts().reindex(range(len(CLASS_NAMES)), fill_value=0)
plt.figure(figsize=(6, 4))
plt.bar(CLASS_NAMES, counts.values)
plt.title("Distribución de clases (balanceada por edad y etnia)")
plt.ylabel("N° de imágenes")
plt.show()

print(counts.rename(index=lambda i: CLASS_NAMES[i]))
# Balanceado por muestreo (cap global segmento×etnia) -> barras del mismo alto.

In [ ]:
# --- Distribución por ETNIA dentro de cada segmento (debe salir perfectamente pareja) ---
tab = (
    df.assign(race_name=df["race"].map(RACE_NAMES))
      .pivot_table(index="label", columns="race_name", aggfunc="size", fill_value=0)
      .reindex(range(len(CLASS_NAMES)))
)
tab.index = [CLASS_NAMES[i] for i in tab.index]
tab = tab.reindex(columns=list(RACE_NAMES.values()), fill_value=0)

print("Imágenes por etnia y segmento (deberían ser todas iguales al cap):\n")
print(tab.to_string())

ax = tab.plot(kind="bar", stacked=True, figsize=(8, 5), colormap="tab10")
ax.set_title("Composición étnica por segmento etario (balanceada, sin 'Others')")
ax.set_ylabel("N° de imágenes")
ax.set_xlabel("Segmento etario")
ax.legend(title="Etnia", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# --- Split train / val / test (estratificado por clase) ---
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

## Parte A — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

class UTKFaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["filepath"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["label"]

train_ds = UTKFaceDataset(train_df, transform)
val_ds = UTKFaceDataset(val_df, transform)
test_ds = UTKFaceDataset(test_df, transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# --- Arquitectura CNN simple ---
class AgeCNN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # IMG_SIZE=64 -> tras 3 poolings de /2: 64 -> 32 -> 16 -> 8
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

model_pt = AgeCNN(num_classes=len(CLASS_NAMES)).to(device)
print(model_pt)

In [ ]:
# --- Entrenamiento (PyTorch) ---
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_pt.parameters(), lr=1e-3)
EPOCHS = 10

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

history_pt = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
t_inicio_pt = time.time()
for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(model_pt, train_loader, train=True)
    va_loss, va_acc = run_epoch(model_pt, val_loader, train=False)
    history_pt["train_loss"].append(tr_loss); history_pt["train_acc"].append(tr_acc)
    history_pt["val_loss"].append(va_loss); history_pt["val_acc"].append(va_acc)
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | val_loss={va_loss:.4f} val_acc={va_acc:.4f}")
t_total_pt = time.time() - t_inicio_pt

print(f"\nTiempo total de entrenamiento (PyTorch): {t_total_pt:.2f} segundos")

In [ ]:
# --- Evaluación en test (PyTorch) ---
model_pt.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model_pt(imgs)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))
cm = confusion_matrix(all_labels, all_preds)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot()
plt.title("Matriz de confusión — PyTorch")
plt.show()

## Parte B — TensorFlow / Keras

In [ ]:
import tensorflow as tf

def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img / 255.0
    return img, label

def make_tf_dataset(dataframe, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((dataframe["filepath"].values, dataframe["label"].values))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds_tf = make_tf_dataset(train_df, shuffle=True)
val_ds_tf = make_tf_dataset(val_df)
test_ds_tf = make_tf_dataset(test_df)

In [ ]:
# --- Arquitectura CNN equivalente, en Keras ---
model_tf = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax"),
])

model_tf.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model_tf.summary()

In [ ]:
# --- Entrenamiento (Keras) ---
EPOCHS_TF = 10
t_inicio_tf = time.time()
history_tf = model_tf.fit(train_ds_tf, validation_data=val_ds_tf, epochs=EPOCHS_TF)
t_total_tf = time.time() - t_inicio_tf

print(f"\nTiempo total de entrenamiento (Keras): {t_total_tf:.2f} segundos")

In [ ]:
# --- Curvas de entrenamiento (Keras) ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history_tf.history["accuracy"], label="train")
axes[0].plot(history_tf.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history_tf.history["loss"], label="train")
axes[1].plot(history_tf.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].legend()
plt.show()

In [ ]:
# --- Evaluación en test (Keras) ---
y_true_tf, y_pred_tf = [], []
for imgs, labels in test_ds_tf:
    preds = model_tf.predict(imgs, verbose=0)
    y_pred_tf.extend(np.argmax(preds, axis=1))
    y_true_tf.extend(labels.numpy())

print(classification_report(y_true_tf, y_pred_tf, target_names=CLASS_NAMES))
cm_tf = confusion_matrix(y_true_tf, y_pred_tf)
ConfusionMatrixDisplay(cm_tf, display_labels=CLASS_NAMES).plot()
plt.title("Matriz de confusión — TensorFlow/Keras")
plt.show()

In [ ]:
# --- Comparación de tiempos de entrenamiento ---
print(f"PyTorch      : {t_total_pt:.2f} s")
print(f"TensorFlow   : {t_total_tf:.2f} s")

## Comparación final

_(Completa aquí: accuracy de cada framework en test, tiempo total de entrenamiento de cada uno (ver celda anterior), qué clases se confunden más entre sí, y qué cambiarías para mejorar el modelo: más datos, augmentación, transfer learning, etc. Nota: al estar balanceado por edad **y** etnia, cualquier diferencia de accuracy entre clases ya no debería explicarse por sesgo étnico.)_